In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))
print(parent_dir)
from _common import DERIV, DST_CRS as TARGET_CRS, path_for
# EPSG =  '6346'
SRC = Path.cwd().parent.parent.parent/"qgis/annotations"
# Still writing to its own file. The scripts now speak this notebook's
# vocabulary (pad_id / pit_inside_id / pit_full), so the two agree -- but the
# live annotations_proj.gpkg is not replaced until the sweep finishes and this
# output has been checked.
OUT = SRC / "annotations_proj_v2.gpkg"

C:\Users\colto\Documents\GitHub\lidar_project\notebooks\wellsight_v2


In [2]:
def loader_function(name: str, *, assume_epsg: str | None = None) -> gpd.GeoDataFrame:
    anno_gdf = gpd.read_file(SRC/f"{name}.shp")
    if anno_gdf.crs is None and assume_epsg is not None:
        # print(f"{name}: no crs is present. Assuming EPSG {assume_epsg}")
        anno_gdf = anno_gdf.set_crs(epsg=assume_epsg)     # assign it

    anno_gdf = anno_gdf.to_crs(TARGET_CRS)
    anno_gdf = anno_gdf[~anno_gdf.geometry.isna() & ~anno_gdf.geometry.is_empty].copy()
    anno_gdf['valid'] = anno_gdf.geometry.is_valid
    bad = (~anno_gdf["valid"]).sum()

    if bad:
        anno_gdf.loc[~anno_gdf['valid'], "geometry"] = anno_gdf.loc[~anno_gdf['valid'], "geometry"].buffer(0)
    anno_gdf = anno_gdf.drop(columns=["valid"])
    return anno_gdf
def assign_pad_id_process(features, pads, geo_type):
    dupe_features_gdf = features.copy()
    if geo_type == 'polygon':
        tester_gdf = gpd.GeoDataFrame(geometry = dupe_features_gdf.geometry.centroid, crs=dupe_features_gdf.crs)
    else:
        tester_gdf = gpd.GeoDataFrame(geometry = dupe_features_gdf.geometry, crs=dupe_features_gdf.crs)
    pad_contain_gdf = gpd.sjoin(tester_gdf, pads[["pad_id", "geometry"]], how="left", predicate="intersects")
    pad_contain_gdf = pad_contain_gdf[~pad_contain_gdf.index.duplicated(keep="first")]
    dupe_features_gdf["pad_id"] = pad_contain_gdf['pad_id'].values
    return dupe_features_gdf




In [3]:
pad_gdf = loader_function("plat")
pit_inside_gdf = loader_function("pit_inside")
pit_full_gdf = loader_function("pit_outside")   # shapefile still named pit_outside.shp
roads_gdf = loader_function("roads")
not_roads_gdf = loader_function("not_roads")
drainage_gdf = loader_function("drainage", assume_epsg=6346)

#assign ID numbers to pads
pad_gdf = pad_gdf.reset_index(drop=True)
pad_gdf['pad_id'] = pad_gdf.index.astype(int)
pad_gdf['area_m2'] = pad_gdf.geometry.area
pad_gdf = pad_gdf.drop(columns=["id"])

In [4]:
pit_inside_gdf = pit_inside_gdf.reset_index(drop=True)
pit_full_gdf = pit_full_gdf.reset_index(drop=True)
pit_inside_gdf["pit_inside_id"] = pit_inside_gdf.index.astype(int)
pit_full_gdf["pit_full_id"] = pit_full_gdf.index.astype(int)

matched_pit_pairs_gdf = gpd.overlay(pit_inside_gdf[["pit_inside_id", "geometry"]], pit_full_gdf[["pit_full_id", "geometry"]], how="intersection", keep_geom_type=True)
matched_pit_pairs_gdf['overlap_area'] = matched_pit_pairs_gdf.geometry.area
matched_pit_pairs_gdf = matched_pit_pairs_gdf.sort_values('overlap_area', ascending=False).drop_duplicates("pit_inside_id")
matched_pair_dict = dict(zip(matched_pit_pairs_gdf["pit_inside_id"], matched_pit_pairs_gdf["pit_full_id"]))



Wall Maker


In [5]:
walls = []
for pid_in, pid_out in matched_pair_dict.items():
    ring = pit_full_gdf.loc[pit_full_gdf.pit_full_id == pid_out, "geometry"].iloc[0].difference(pit_inside_gdf.loc[pit_inside_gdf.pit_inside_id == pid_in, "geometry"].iloc[0])
    if not ring.is_empty:
        walls.append({"pit_inside_id": pid_in, "geometry":ring})
pit_wall_gdf = gpd.GeoDataFrame(walls, crs=TARGET_CRS)

ID Assigner

In [6]:
# The ring's reference to its inner pit is a foreign key into pit_inside, so
# it carries that layer's id column name -- the same rule pad_id follows, and
# the same name cell 5 already uses on pit_wall. All three pit layers now
# join on pit_inside_id.
flipped_matched_pair_dict ={v: k for k, v in matched_pair_dict.items()}
pit_full_gdf["pit_inside_id"] = pit_full_gdf["pit_full_id"].map(flipped_matched_pair_dict)
pit_full_gdf = assign_pad_id_process(pit_full_gdf, pad_gdf, "polygon")
pit_inside_gdf = assign_pad_id_process(pit_inside_gdf, pad_gdf, "polygon")
pit_wall_gdf = assign_pad_id_process(pit_wall_gdf, pad_gdf, "polygon")
roads_gdf  = assign_pad_id_process(roads_gdf , pad_gdf, "line")
not_roads_gdf  = assign_pad_id_process(not_roads_gdf , pad_gdf, "line")
drainage_gdf  = assign_pad_id_process(drainage_gdf , pad_gdf, "line")


Column edits

In [7]:
pit_per_pad_gdf = pit_inside_gdf.dropna(subset=["pad_id"]).groupby("pad_id").size().rename("n_pits")
road_per_pad_gdf = roads_gdf.dropna(subset=["pad_id"]).groupby("pad_id").size().rename("n_roads")
not_road_per_pad_gdf = not_roads_gdf.dropna(subset=["pad_id"]).groupby("pad_id").size().rename("n_not_roads")

# Drop any tally columns left by a previous run first. Without this, re-running
# the cell merges onto an already-merged frame and pandas appends n_pits_x /
# n_pits_y instead of replacing -- which is what happened to annotations_proj.gpkg.
pad_gdf = pad_gdf.drop(columns=[c for c in pad_gdf.columns
                                if c.startswith(("n_pits", "n_roads", "n_not_roads"))])
pad_gdf = (pad_gdf.merge(pit_per_pad_gdf, on="pad_id", how="left")
                  .merge(road_per_pad_gdf, on="pad_id", how="left")
                  .merge(not_road_per_pad_gdf, on="pad_id", how="left"))
for col in ("n_pits", "n_roads", "n_not_roads"):
    pad_gdf[col] = pad_gdf[col].fillna(0).astype(int)

Index(['id', 'geometry', 'pad_id', 'area_m2', 'n_pits', 'n_roads',
       'n_not_roads'],
      dtype='str')


In [8]:
if OUT.exists():
    OUT.unlink()                         # delete the old file so we never half-overwrite it
pad_gdf.to_file(OUT, layer="pad", driver="GPKG")  # a GeoPackage holds many layers in one file
pit_inside_gdf.to_file(OUT, layer="pit_inside", driver="GPKG")    # each call appends another layer
pit_full_gdf.to_file(OUT, layer="pit_full", driver="GPKG")
pit_wall_gdf.to_file(OUT, layer="pit_wall", driver="GPKG")
roads_gdf.to_file(OUT, layer="roads", driver="GPKG")
not_roads_gdf.to_file(OUT, layer="not_roads", driver="GPKG")
drainage_gdf.to_file(OUT, layer="drainage", driver="GPKG")